In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys; sys.path.append("../generators")
from cdc_simulator import write_batch, describe
describe()

In [0]:
#generate first batch

write_batch(1)

In [0]:
%sql

CREATE OR REPLACE TEMPORARY TABLE  tmp_customers AS 
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/customers/batch_01.json",
    format => "json");

DESCRIBE EXTENDED tmp_customers;

In [0]:
%sql

CREATE OR REPLACE TEMPORARY TABLE  tmp_accounts AS 
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/accounts/batch_01.json",
    format => "json");

DESCRIBE EXTENDED tmp_accounts;

In [0]:
%sql
SELECT count(*) FROM tmp_customers;
SELECT count(*) FROM tmp_accounts;

In [0]:
#Launch batch 2

write_batch(2)

In [0]:
%sql
---check status of the silver customer table

SELECT customer_id, count(*) AS n_versions, count_if(__END_AT IS NULL) AS n_current
FROM bank.silver.customers
GROUP BY ALL
HAVING count(*) > 1 OR count_if(__END_AT IS NULL) = 0
ORDER BY 1;



In [0]:
%sql
SELECT
*
FROM  bank.silver.customers_current
WHERE customer_id = 'C001';

In [0]:
# write 3rd batch 
write_batch(3)

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/customers/batch_03.json",
    format => "json");

In [0]:
%sql
SELECT customer_id, address, __START_AT, __END_AT
FROM bank.silver.customers WHERE customer_id = 'C004' ORDER BY __START_AT;

In [0]:
%sql
SELECT *
FROM read_files(
    "/Volumes/bank/ops/landing/cdc/accounts/batch_03.json",
    format => "json");